# 💊 RefillCare: Medication Refill Reminder System
## End-to-End Walkthrough: Phases 1, 2, and 3

**Author:** RefillCare Core Pipeline  
**Date:** 2026-09-07  

---

### 🎯 Project Overview
RefillCare is an intelligent pharmacy refill reminder system built to predict when patients will need chronic medication refills based on historical purchasing intervals and active pharmaceutical ingredient (SALT) patterns.

### 📚 Notebook Contents
1. **Phase 1 — Exploratory Data Analysis & Discovery:** Raw transaction inspection, customer identity resolution, shared-phone analysis, and SALT catalog mapping.
2. **Phase 2 — Data Pipeline & History Creation:** Data cleaning, customer ID sanitization, invoice line aggregation, SALT enrichment, and chronological interval calculation.
3. **Phase 3 — Leakage-Safe Feature Engineering & Baseline:** Supervised target formulation ($y = t_{i+1} - t_i$), 36 engineered features, temporal splitting, and historical median baseline evaluation.

## 🛠️ Step 0: Imports & Universal Path Resolver

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import json
try:
    from IPython.display import display
except ImportError:
    display = print
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

current_dir = Path(__file__).resolve().parent if '__file__' in locals() else Path.cwd()
project_root = current_dir.resolve()
while project_root.parent != project_root and not (project_root / 'refillcare' / '__init__.py').exists():
    project_root = project_root.parent
if (project_root / 'refillcare' / '__init__.py').exists() and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

def find_file(relative_path: str) -> Path:
    candidates = [
        project_root / relative_path,
        Path.cwd() / relative_path,
        Path('..') / relative_path,
        Path('../..') / relative_path,
        Path(r'C:\Users\sunil\ai-mediastra-whatsapp-reminder\ai-mediastra-whatsapp-reminder') / relative_path,
    ]
    for c in candidates:
        if c.exists():
            return c.resolve()
    return candidates[0]
print(f'[PASS] Project Root resolved: {project_root}')


---
# 🔍 Phase 1: Exploratory Data Analysis & Source Inspection

Phase 1 analyzed the raw pharmacy point-of-sale dataset (`customer_data_fields.csv`) and master catalog (`SALT WISE ITEMS.xlsx`).

### Key Discoveries from Phase 1:
- **895,557 raw transaction rows** across ~5.8 years (`2020-12-24` to `2026-08-31`).
- **Identity Resolution:** `customerId` (28,413 unique) is the unique patient identifier. Phone numbers (`MOBILE_NO`) are shared across family members (136 shared phones involving 341 customers) and must NOT be used as customer identity.
- **Date Parsing:** Dates are formatted as `DD/MM/YYYY`, requiring `dayfirst=True`.
- **Duplicate Lines:** Multiple rows per `(customerId + invoice_number + itemId)` represent split batches on the same visit and must be aggregated.

In [ ]:
# 1.1 Load and inspect raw transactions
raw_csv_path = find_file('data/refillcare/customer_data_fields.csv')
print(f'Loading from: {raw_csv_path}')

if raw_csv_path.exists():
    raw_sample = pd.read_csv(raw_csv_path, nrows=5000)
    print(f'✅ Loaded {len(raw_sample):,} rows sample')
    display(raw_sample[['invoice_date', 'invoice_number', 'customerId', 'customerName', 'itemId', 'itemName', 'quantity', 'MOBILE_NO']].head(5))
else:
    print(f'❌ File not found at: {raw_csv_path}')

In [ ]:
# 1.2 Inspect SALT Master Catalog
salt_excel_path = find_file('data/refillcare/SALT WISE ITEMS.xlsx')
print(f'Loading from: {salt_excel_path}')

if salt_excel_path.exists():
    salt_sample = pd.read_excel(salt_excel_path, sheet_name='Rate List', skiprows=5, nrows=10, engine='openpyxl')
    print(f'✅ SALT Master Catalog Sample (skiprows=5):')
    display(salt_sample[['Code', 'Item Name', 'PACK', 'SALT', 'CATEGORY', 'ITEMCAT']].head(5))
else:
    print(f'❌ File not found at: {salt_excel_path}')

---
# 🧹 Phase 2: Data Pipeline, History Creation & SALT Enrichment

Phase 2 built a reusable, tested pipeline (`refillcare.data`) that executes:
1. **Cleaning & Sanitization:** Removes exact duplicates, drops null `mfgDate`, and sanitizes noisy characters from `customerId`.
2. **Invoice Aggregation:** Sums quantities and amounts for duplicate invoice lines, creating 1 purchase event per visit.
3. **SALT Enrichment:** Left joins master catalog active ingredients on `Code` $\leftrightarrow$ `itemCode`.
4. **History & Interval Math:** Constructs chronological timelines per `customerId + itemId` and computes backward-looking intervals.

In [ ]:
# 2.1 Load Phase 2 Processed Purchase History
history_parquet_path = find_file('data/refillcare/processed/purchase_history.parquet')
print(f'Loading from: {history_parquet_path}')

if history_parquet_path.exists():
    history_df = pd.read_parquet(history_parquet_path)
    print(f'✅ Total Purchase Events: {len(history_df):,}')
    print(f'✅ Unique Customer-Medicine Histories: {history_df.groupby(["customerId", "itemId"]).ngroups:,}')
    display(history_df[['customerId', 'customerName', 'itemId', 'itemName', 'invoice_date', 'quantity', 'purchase_seq', 'days_since_previous_purchase', 'salt_composition']].head(8))
else:
    print(f'❌ File not found at: {history_parquet_path}')

In [ ]:
# 2.2 Inspect Customer Refill Interval Statistics
intervals = history_df['days_since_previous_purchase'].dropna()

print('--- Refill Interval Summary ---')
print(f'Total Intervals:    {len(intervals):,}')
print(f'Median Interval:    {intervals.median():.1f} days')
print(f'Mean Interval:      {intervals.mean():.2f} days')
print(f'Std Deviation:      {intervals.std():.2f} days')
recurring_pct = ((intervals >= 15) & (intervals <= 120)).sum() / len(intervals) * 100
print(f'Recurring Cycles (15-120d): {recurring_pct:.2f}%')

---
# 🚀 Phase 3: Feature Engineering & Supervised Learning Dataset

Phase 3 constructed the leakage-safe training dataset (`refillcare.features`) and established baseline benchmark performance.

### Supervised Learning Formulation:
- **Target:** `target_days_until_next_purchase` = $t_{i+1} - t_i$.
- **Final Purchase Rule:** Event $N$ in every history has no known next purchase and is strictly excluded from training targets.
- **36 Leakage-Safe Features:** Historical expanding median/mean/std, quantity ratios, calendar features, and medicine/SALT attributes.

In [ ]:
# 3.1 Load Processed Supervised Datasets
train_path = find_file('data/refillcare/processed/train.parquet')
val_path = find_file('data/refillcare/processed/validation.parquet')
test_path = find_file('data/refillcare/processed/test.parquet')

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)
test_df = pd.read_parquet(test_path)

print(f'✅ Train Set:      {len(train_df):,} rows ({train_df.invoice_date.min().date()} to {train_df.invoice_date.max().date()})')
print(f'✅ Validation Set: {len(val_df):,} rows ({val_df.invoice_date.min().date()} to {val_df.invoice_date.max().date()})')
print(f'✅ Test Set:       {len(test_df):,} rows ({test_df.invoice_date.min().date()} to {test_df.invoice_date.max().date()})')

# Display engineered features sample
display(train_df[['customerId', 'itemId', 'invoice_date', 'purchase_count_so_far', 'historical_interval_median', 'avg_historical_quantity', 'purchase_month', 'target_days_until_next_purchase']].head(8))

In [ ]:
# 3.2 Inspect Phase 3 Quality Report JSON
report_json_path = find_file('data/refillcare/processed/phase3_quality_report.json')
if report_json_path.exists():
    with open(report_json_path, 'r') as f:
        report = json.load(f)
    
    print('=== Phase 3 Baseline Benchmark Results ===')
    print(f'Fallback Training Median: {report["baseline_evaluation"]["fallback_median_days"]} days')
    print('\nValidation Metrics:')
    for k, v in report["baseline_evaluation"]["validation"].items():
        if k != 'target_summary':
            print(f'  - {k}: {v}')
            
    print('\nTest Metrics:')
    for k, v in report["baseline_evaluation"]["test"].items():
        if k != 'target_summary':
            print(f'  - {k}: {v}')

---
## 🏁 Summary & Phase 4 Roadmap

| Phase | Deliverable | Status |
|---|---|---|
| **Phase 1** | Data Analysis & Quality Profiling (`docs/PHASE_1_REFILLCARE_DATA_ANALYSIS.md`) | ✅ Complete |
| **Phase 2** | Data Cleaning, Aggregation & History Pipeline (`refillcare.data`) | ✅ Complete |
| **Phase 3** | Leakage-Safe Feature Engineering & Baseline (`refillcare.features`) | ✅ Complete |
| **Phase 4** | **Machine Learning Model Training (XGBoost / LightGBM)** | 🔜 Ready to Begin |

### Next Step: Phase 4
Train Gradient Boosted Decision Trees (XGBoost / LightGBM) and Quantile Regressors on `train.parquet`, validate on `validation.parquet`, and evaluate on `test.parquet` to beat the baseline MAE of 19.2 days.